# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [158]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [159]:
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1357.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [160]:
query_embeddings.shape

(5346, 384)

In [161]:
query_embeddings[:5]


array([[-0.02074328,  0.08789469,  0.06243357, ..., -0.03356438,
        -0.04759217, -0.03279905],
       [ 0.00096978, -0.01984177, -0.04080513, ...,  0.02996143,
        -0.02697211,  0.10606926],
       [-0.02252915,  0.0716615 ,  0.07225666, ..., -0.09477419,
        -0.04195307, -0.01623018],
       [-0.0236518 ,  0.02925191,  0.01507207, ..., -0.06504929,
         0.02110572,  0.04893175],
       [ 0.01233931, -0.03880871, -0.00399657, ...,  0.09457004,
        -0.06981922, -0.02656643]], shape=(5, 384), dtype=float32)

In [162]:
def create_embedding(text):
    return model.encode(text, convert_to_numpy=True).astype(np.float32)

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [163]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [164]:
import redis

r = redis.Redis(host='localhost', port=6379)

# Ejecutar antes docker run --rm --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest

# **Objetivo I**

### **Tarea 1**. Buscar la estructura de datos más apropiada para la cache.

Para esta tarea, la estructura de datos más apropiada para la caché en Redis sería una **tabla Hash**. Esto es ideal para almacenar la información de cada carta, ya que cada carta puede tener múltiples atributos (nombre, texto, tipo, etc.) que pueden ser almacenados como campos dentro del Hash.  Además, los Hashes permiten un acceso rápido a los datos, lo que es crucial para una caché eficiente. Así, tenemos un Hash para cada carta con el nombre de la carta como clave y los atributos como campos dentro del Hash.


### **Tarea 2**. Escribir una función Python que reciba un fichero .csv con un conjunto de ejemplo de cartas y las cargue en una base de datos Redis usando la estructura de datos seleccionada en el paso anterior.


In [165]:
df = pd.read_csv("cards_final_with_xp.csv")

In [166]:
df.head()

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic


In [167]:
def cargar_cartas_en_redis(df: pd.DataFrame) -> None:
    """
    Carga las cartas de un DataFrame en Redis.

    Args:
        df (pd.DataFrame): DataFrame que contiene las cartas.
    """
    for _, serie in df.iterrows():

        data = serie.to_dict()
        code = data.pop("code", None)
        
        if code:
            key = f"card:{code}"
            
        r.hset(key, mapping=data)

In [168]:
cargar_cartas_en_redis(df)

### **Tarea 3**. Escribir funciones python que permita realizar cada una de las acciones. Implementar una por acción.

#### 1. Saber si una carta está en la cache por su campo code.

In [169]:
def carta_en_cache(code: str) -> bool:
    """
    Verifica si una carta está en la cache por su campo code.

    Args:
        code (str): El código de la carta a verificar.

    Returns:
        bool: True si la carta está en la cache, False en caso contrario.
    """
    key = f"card:{code}"
    return r.exists(key) == 1

In [170]:
carta_en_cache("01001")

False

In [171]:
carta_en_cache("06095")

True

#### 2. Recuperar todos los datos de una carta a partir de su code.

In [172]:
def recuperar_carta(code: str) -> bool:
    """
    Recupera todos los datos de una carta a partir de su code.
    
    Args:
        code (str): El código de la carta a recuperar.
        
    Returns:
        dict: Un diccionario con los datos de la carta si está en la cache, None en caso contrario.
    """
    key = f"card:{code}"
    
    if not r.exists(key):
        return None

    data = r.hgetall(key)
    
    return {
        k.decode("utf-8", "ignore"): v.decode("utf-8", "ignore")
        for k, v in data.items()
    }

In [173]:
recuperar_carta("06095")

{'image_url': 'https://arkhamdb.com/bundles/cards/06095.jpg',
 'xp': '0',
 'pack_code': 'tde',
 'type_code': 'treachery',
 'name': 'Deeper Slumber',
 'illustrator': 'Sacha Angel Diener',
 'text': 'Revelation - Put Deeper Slumber into play in your threat area. Your maximum hand size is reduced by 3 and is checked after each time you draw 1 or more cards. [action] [action]: Discard Deeper Slumber.',
 'embedding': '@<\x15<<\'S\x1f\x1f<u\x15<Ҋ7<?=T\x00ؼ<ƛI\x13X<9=Yh\x0f<66;B=\x06L&<*\x18>&\x05j/;\x14(<\ncy:YHy<xּֽ\\\x1c>:\x01=T+\x15h\x1enRYɼ4=e*=3\x04<\x16,g<=\x14\x03g^=t46"\x19(R=9X<g<aT=!\U000a6f50<(<d"=<\x1dǄ<q<c<JN[OQ@V=\x00\x13P=\x00B\x06=Z\x1d"=sG\x1f[\x08<!\t~<:<\x1fc<\x14=\x14:g]n]<-\x18+<v3iJI\'e->=z\x14d\x15BϽ\u05fa9C)<R=%<\x0c=ը<r=\x14Þ;Hư\'ڼeL<2e<\x11;!=Ўz=Q=Z=<\n=Zש=s<U=<4\x0b>ׁx!({=z\x10\x97\tcy*==\x01/\x11t\x16=\x0b=o\x1c\x01=6;ט=u֣<\x7f<(=pP@N,\n(=%࠼aVFA:J<G#6<6\x18<\x13\x16bI:<-;Ѽύ2i%<cdj3<1e=\r<\x16\x07f/Tp;=U|6\n\x7fջVha;P=\u05cbG\x18ܽ\x13G=L%=]\x0f|\x08\x1a\x1d\x00H\r\x

#### 3. Meter una carta nueva en la cache.


In [174]:
def meter_carta_en_cache(code: str, data: dict) -> None:
    """
    Mete una carta nueva en la cache si no existe ya.

    Args:
        code (str): El código de la carta a meter.
        data (dict): Un diccionario con los datos de la carta a meter.
    """
    key = f"card:{code}"
    r.hset(key, mapping=data) if r.exists(key) == 0 else print(f"La carta con código {code} ya existe en la cache.")

In [175]:
data_example = {
    "name": "Test Card",
    "text": "This is a test card.",
    "type_code": "test_type",
    "traits": "test_traits",
    "pack_code": "test_pack",
    "faction_code": "test_faction",
    "xp": 0,
    "illustrator": "test_illustrator",
    "image_url": "http://example.com/test_card.jpg"
}

In [176]:
meter_carta_en_cache("01001", data_example)

In [177]:
recuperar_carta("01001")

{'name': 'Test Card',
 'text': 'This is a test card.',
 'type_code': 'test_type',
 'traits': 'test_traits',
 'pack_code': 'test_pack',
 'faction_code': 'test_faction',
 'xp': '0',
 'illustrator': 'test_illustrator',
 'image_url': 'http://example.com/test_card.jpg'}

#### 4. Eliminar una carta de la cache a partir de su campo code.

In [178]:
def eliminar_carta_de_cache(code: str) -> None:
    """
    Elimina una carta de la cache a partir de su campo code.

    Args:
        code (str): El código de la carta a eliminar.
    """
    key = f"card:{code}"
    r.delete(key) if r.exists(key) == 1 else print(f"La carta con código {code} no existe en la cache.")
    

In [179]:
eliminar_carta_de_cache("01001")

In [180]:
recuperar_carta("01001")

# **Objetivo II** 

El equipo del portal ha oído hablar de las capacidades de búsqueda avanzada de Redis y quiere probar si se pueden usar para extender la funcionalidad del portal. En concreto han identificado varias búsquedas que han reclamado los usuarios a lo largo de los años:

- **A**. En el juego hay 7 facciones, 5 que son clases que pueden usar los jugadores (mystic, survivor, guardian, seeker y rogue), la facción “neutral” que es equipo común para todas las clases y la facción “mythos” que son los enemigos. Recientemente se han añadido cartas especiales que tienen más de una facción. Los jugadores están interesados en buscar todas las cartas que contengan varias facciones a la vez, así como, buscar todas las cartas que contengan al menos una de las facciones que indiquen. Por defecto devolvemos las cartas de 5 en 5.
- **B**. Los traits son una forma muy cómoda de buscar cartas interesantes y es muy usada por los
usuarios. No obstante, como hay muchos, manejarlos es algo complicado. Los usuarios
quieren saber cuáles son los traits más comunes para su facción por orden de frecuencia.
Por defecto mostramos páginas con 15 traits.

- **C**. Durante el juego los jugadores ganan puntos de experiencia y pueden gastarlos en mejorar sus mazos. Por eso, es normal que los usuarios quieran buscar cartas que contengan ciertos traits y que puedan usarse para actualizar sus cartas, es decir, que tengan un coste de experiencia (xp) > 0. Normalmente están interesados en cartas que sean de su facción, así que quieren que esas aparezcan entre los primeros resultados, no obstante, también quieren ver cartas de otras facciones ya que hay mazos que mezclan varias facciones. No obstante, nunca quieren ver cartas de la facción “mythos” porque no las pueden incluir en su mazo. Los jugadores también piden filtrar las cartas por lo puntos de experiencia que les quedan, para que no les aparezcan cartas que no pueden comparar. Por defecto devolvemos las cartas de 5 en 5.


### Tarea 1. Diseña un indice que permita realizar las búsquedas anteriores.

Para crear este indice, tenemos que tener en cuenta lo siguiente:

- Como vamos a realizar búsquedas por facción, es importante tener un indice que nos permita buscar por este campo, que sera de tipo TAG debido a que los valores son cadenas de texto concretas. Ademas de esto, como las cartas pueden tener varias facciones, es necesario agregar un separador para poder almacenar varias facciones en el mismo campo. En este caso, tendremos que usar el separador “|” para separar las facciones en el campo de facciones. 

- Para la búsqueda por traits, es importante tener un indice que nos permita buscar por este campo, que sera de tipo TAG debido a que los valores son cadenas de texto concretas. Ademas de esto, como las cartas pueden tener varios traits, es necesario agregar un separador para poder almacenar varios traits en el mismo campo. En este caso, tendremos que usar el separador “|” para separar los traits en el campo de traits.

- Para la búsqueda por coste de experiencia, es importante tener un indice que nos permita buscar por este campo, que sera de tipo NUMERIC debido a que los valores son números enteros. Ademas de esto, como queremos ordenarlos por coste de experiencia, es necesario agregar un SORTABLE para poder ordenar los resultados por este campo.

In [181]:
from redis.exceptions import ResponseError

def crear_indice(nombre_indice: str) -> None:
    """
    Crea un indice en Redis para las cartas.
    """
    try:
        comando = f"""
        FT.CREATE {nombre_indice} 
        ON HASH PREFIX 1 card: 
        SCHEMA faction_code TAG SEPARATOR | 
        traits TAG SEPARATOR | 
        xp NUMERIC SORTABLE
        """
        
        r.execute_command(*comando.split())
        print(f"Índice '{nombre_indice}' creado con éxito.")
        
    except ResponseError as e:
        if "Index already exists" in str(e):
            print(f"El índice '{nombre_indice}' ya existe. No es necesario crearlo de nuevo.")
        else:
            print(f"Error de Redis al crear el índice: {e}")
            
    except Exception as e:
        print(f"Error inesperado: {e}")


In [182]:
crear_indice("cards-idx")

El índice 'cards-idx' ya existe. No es necesario crearlo de nuevo.


Para cubrir las tres búsquedas del **Objetivo II**, el índice debe modelar cada campo según su uso:

- **A) Búsquedas AND/OR por facciones (incluyendo multifacción)**  
    `faction_code` debe definirse como **TAG** con `SEPARATOR "|"`, porque una carta puede tener varias facciones en el mismo campo (por ejemplo: `guardian|seeker`).  
    - **OR**: permite recuperar cartas que tengan *al menos una* facción solicitada.  
    - **AND**: permite exigir que una carta tenga *todas* las facciones indicadas.

- **B) Traits más comunes por facción**  
    `traits` también debe ser **TAG** con `SEPARATOR "|"`, ya que contiene múltiples valores.  
    Esto habilita agregaciones con **FT.AGGREGATE** (`GROUPBY` + `REDUCE COUNT`) para contar frecuencia de cada trait dentro de una facción y ordenarlos por popularidad.

- **C) Filtros de mejoras por experiencia y exclusiones**  
    `xp` debe ser **NUMERIC** para consultas por rango (`xp > 0`, `xp <= xp_max`).  
    Si además se quiere ordenar resultados por coste de experiencia, conviene marcarlo como **SORTABLE**.  
    La exclusión de facciones (por ejemplo `mythos`) se resuelve con filtros negativos sobre el campo TAG de facción, y la paginación con `LIMIT offset num`.

1) Índice (FT.CREATE)
Dado que las cartas están en HASH con un prefijo card: (ej.: card:1029) y que dentro del hash existen los campos del enunciado.

In [183]:
import redis
from redis.exceptions import ResponseError

def init_redis(host="localhost", port=6379, db=0) -> redis.Redis:
    # decode_responses=True para trabajar con strings (más cómodo en notebooks)
    return redis.Redis(host=host, port=port, db=db, decode_responses=True)

def ensure_cards_index(r: redis.Redis, index_name="cards-idx", prefix="card:", drop=False) -> None:
    """
    Crea el índice si no existe.
    - faction_code y traits son TAG multi-valor separados por '|'
    - xp es NUMERIC y SORTABLE para filtrar y (opcionalmente) ordenar.
    """
    if drop:
        try:
            r.execute_command("FT.DROPINDEX", index_name, "DD")
        except ResponseError:
            pass

    # Si existe, no hacemos nada
    try:
        r.execute_command("FT.INFO", index_name)
        return
    except ResponseError:
        pass

    # FT.CREATE ON HASH PREFIX ... SCHEMA ...
    # Sintaxis base: :contentReference[oaicite:4]{index=4}
    r.execute_command(
        "FT.CREATE", index_name,
        "ON", "HASH",
        "PREFIX", "1", prefix,
        "SCHEMA",
        # Campos clave para Objetivo II
        "code", "TAG",
        "name", "TEXT",
        "text", "TEXT",
        "type_code", "TAG",
        "pack_code", "TAG",
        "faction_code", "TAG", "SEPARATOR", "|",   # multi-facción 
        "traits", "TAG", "SEPARATOR", "|",         # multi-traits 
        "xp", "NUMERIC", "SORTABLE",               # rangos/ordenación 
        # Campos extra (no imprescindibles para Obj II, pero útiles para RETURN)
        "illustrator", "TEXT",
        "image_url", "TEXT"
    )


* Usamos TAG SEPARATOR | porque el enunciado dice que traits y faction_code vienen separados por | si hay más de uno, y los PDFs explican que TAG multi-valor se soporta con SEPARATOR.
* FT.SEARCH soporta paginación con LIMIT offset n y devolver campos con RETURN.

### Tarea 2. Implementa una función python que realize cada una de las búsquedas anteriores

#### Helpers para parsear respuestas

In [184]:
def _parse_ft_search(raw):
    """
    Parse para FT.SEARCH:
    RESP2: [total, key1, [field, value, ...], key2, [...], ...]
    (y tolera raw=None)
    Devuelve: (total:int, docs:list[dict]) con strings.
    """
    if raw is None:
        return 0, []

    # Si algún día te devolviera dict (RESP3), lo dejamos como fallback
    if isinstance(raw, dict):
        total = int(raw.get("total", 0))
        docs = raw.get("documents", [])
        return total, docs

    def _dec(x):
        return x.decode() if isinstance(x, (bytes, bytearray)) else x

    total = int(raw[0])
    out = []
    i = 1
    while i + 1 < len(raw):
        key = _dec(raw[i])              # "card:1029"
        fields = raw[i + 1]             # [k,v,k,v,...] (bytes normalmente)
        doc = {"_key": key}

        # fields puede venir como list/tuple
        if isinstance(fields, (list, tuple)):
            for j in range(0, len(fields) - 1, 2):
                k = _dec(fields[j])
                v = _dec(fields[j + 1])
                doc[k] = v

        # FIX CLAVE: si no hay 'code' en el HASH/RETURN, sacarlo del docid
        if not doc.get("code"):
            doc["code"] = key.split(":", 1)[1] if ":" in key else key

        out.append(doc)
        i += 2

    return total, out

#### **Objetivo II-A**: búsqueda por facciones

En el enunciado se especifica que quieren: 

- Buscar todas las cartas que contengan varias facciones a la vez (AND): tienen que contener las facciones indicadas, pero pueden contener más. Por ejemplo, si se buscan cartas con las facciones “guardian” y “seeker” en modo AND, se devolverán cartas que tengan ambas facciones, aunque también puedan tener otras facciones adicionales (por ejemplo, una carta con facciones “guardian|seeker|neutral” también sería válida).

- Buscar todas las cartas que contengan al menos una de las facciones que indiquen (OR): tienen que contener al menos una de las facciones indicadas, pero pueden contener más. Por ejemplo, si se buscan cartas con las facciones “guardian” y “seeker” en modo OR, se devolverán cartas que tengan al menos una de esas facciones (por ejemplo, una carta con facciones “guardian|neutral” o “seeker|mythos” también sería válida o incluso alguna con ambas).

In [185]:
def search_by_factions(
    r: redis.Redis,
    factions: list[str],
    mode: str = "OR",            # "OR" o "AND"
    page: int = 0,
    page_size: int = 5,
    index_name: str = "cards-idx",
):
    """
    Devuelve cartas MULTIFACCIÓN (estrictamente más de una facción) que:
    - OR: tengan al menos una de las facciones y al menos otra diferente.
    - AND: tengan todas las facciones (si es solo una, obliga a tener otra).
    Paginado de 5 en 5 (por defecto).
    """
    if not factions:
        raise ValueError("factions no puede estar vacío")

    factions = [f.strip().lower() for f in factions]
    
    # Facciones válidas para multifacción (se excluyen 'neutral' y 'mythos')
    valid_factions = ["mystic", "survivor", "guardian", "seeker", "rogue"]

    if mode.upper() == "AND":
        if len(factions) == 1:
            # Si piden 1 sola, exigimos que tenga esa AND alguna de las demás válidas
            fac = factions[0]
            others = [f for f in valid_factions if f != fac]
            query = f"@faction_code:{{{fac}}} @faction_code:{{{'|'.join(others)}}}"
        else:
            # Si piden 2 o más con AND, ya es obligatoriamente multifacción por definición
            query = " ".join([f"@faction_code:{{{f}}}" for f in factions])
    else:
        # Modo OR: Para cada facción pedida, construimos la condición de que sea multifacción
        or_blocks = []
        for fac in factions:
            others = [f for f in valid_factions if f != fac]
            # Bloque: (Tiene la facción actual AND tiene al menos una de las otras)
            block = f"(@faction_code:{{{fac}}} @faction_code:{{{'|'.join(others)}}})"
            or_blocks.append(block)
        
        # Unimos todos los bloques con OR lógico a nivel de query
        query = " | ".join(or_blocks)

    offset = page * page_size

    # Ejecutamos el comando manteniendo tu RETURN y LIMIT
    raw = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "4", "code", "name", "faction_code", "xp",
        "LIMIT", str(offset), str(page_size)
    )
    
    # Asumo que _parse_ft_search ya la tienes implementada por tu cuenta
    total, docs = _parse_ft_search(raw)
    
    return {
        "total": total, 
        "page": page, 
        "page_size": page_size, 
        "results": docs, 
        "query": query
    }

In [186]:
search_by_factions(r, ["guardian", "seeker"], mode="or", page=0, page_size=5)

{'total': 34,
 'page': 0,
 'page_size': 5,
 'results': [{'_key': 'card:5115',
   'name': '.45 Thompson',
   'faction_code': 'guardian|rogue',
   'xp': '0',
   'code': '5115'},
  {'_key': 'card:5116',
   'name': 'Scroll of Secrets',
   'faction_code': 'seeker|mystic',
   'xp': '0',
   'code': '5116'},
  {'_key': 'card:5118',
   'name': 'Enchanted Blade',
   'faction_code': 'mystic|guardian',
   'xp': '0',
   'code': '5118'},
  {'_key': 'card:5119',
   'name': 'Grisly Totem',
   'faction_code': 'survivor|seeker',
   'xp': '0',
   'code': '5119'},
  {'_key': 'card:08083',
   'name': 'Medical Student',
   'faction_code': 'guardian|seeker',
   'xp': '0',
   'code': '08083'}],
 'query': '(@faction_code:{guardian} @faction_code:{mystic|survivor|seeker|rogue}) | (@faction_code:{seeker} @faction_code:{mystic|survivor|guardian|rogue})'}

### Objetivo II-B: traits más comunes por facción

4) Objetivo II-B: traits más comunes por facción (frecuencia), páginas de 15
Aquí lo natural es FT.AGGREGATE con GROUPBY + REDUCE COUNT, tal como aparece en el PDF de agregaciones.
Como traits viene “aplanado” en un string con |, hay 2 enfoques:
B1) (Preferido) FT.AGGREGATE con split + UNWIND (si tu RediSearch lo soporta)

In [187]:
def top_traits_for_faction(
    r,
    faction: str,
    page: int = 0,
    page_size: int = 15,
    index_name: str = "cards-idx",
):
    faction = faction.strip().lower()
    offset = page * page_size

    query = f"@faction_code:{{{faction}}}"

    try:
        raw = r.execute_command(
            "FT.AGGREGATE", index_name, query,
            "LOAD", "1", "traits",
            "APPLY", "split(@traits, '|')", "AS", "trait",
            "GROUPBY", "1", "@trait",
            "REDUCE", "COUNT", "0", "AS", "frecuencia",
            "SORTBY", "2", "@frecuencia", "DESC",
            "LIMIT", str(offset), str(page_size)
        )

        total_groups = raw[0]
        results = []

        for i in range(1, len(raw)):
            row_data = raw[i]
            
            res_dict = {}
            for j in range(0, len(row_data), 2):
                key = row_data[j].decode('utf-8') if isinstance(row_data[j], bytes) else row_data[j]
                val = row_data[j+1].decode('utf-8') if isinstance(row_data[j+1], bytes) else row_data[j+1]
                res_dict[key] = val
            
            t_name = res_dict.get("trait")
            
            if t_name and t_name.strip():
                results.append((t_name.strip(), int(res_dict.get("frecuencia", 0))))

        return {
            "faction": faction,
            "total_traits": total_groups,
            "page": page,
            "page_size": page_size,
            "results": results,
            "method": "FT.AGGREGATE"
        }

    except Exception as e:
        print(f"Error ejecutando FT.AGGREGATE: {e}")
        return None

In [188]:
top_traits_for_faction(r, "survivor", page=0, page_size=15)

{'faction': 'survivor',
 'total_traits': 64,
 'page': 0,
 'page_size': 15,
 'results': [('Item', 59),
  ('Fortune', 29),
  ('Ally', 25),
  ('Blessed', 25),
  ('Tool', 22),
  ('Innate', 22),
  ('Talent', 21),
  ('Spirit', 18),
  ('Weapon', 17),
  ('Melee', 16),
  ('Tactic', 16),
  ('Charm', 14),
  ('Cursed', 13),
  ('Trick', 13),
  ('Developed', 8)],
 'method': 'FT.AGGREGATE'}

B2) (Fallback robusto) FT.SEARCH + contar en Python
Sigue usando RediSearch para filtrar por facción, y luego agregas tú. Si el dataset es el del curso, suele ser manejable.

5) Objetivo II-C: buscar cartas con xp>0, excluir mythos, filtrar por xp máximo, priorizar facción y permitir mezcla
El enunciado pide:
xp > 0
excluir mythos siempre
filtrar por xp máximo
mostrar “preferidas” de una facción al principio, pero permitir otras facciones después
páginas de 5 
practica_redis_busqueda_hibrida…
Para que sea simple y 100% controlable, lo hago en 2 consultas:
primero facción preferida
luego el resto (sin duplicados)
y mezclo manteniendo el orden “preferida primero”.

In [189]:
def search_upgrades(
    r,
    traits=None,
    preferred_faction=None,
    xp_max=None,
    page=0,
    page_size=5,
    index_name="cards-idx",
):
    def _to_str(v):
        return v.decode("utf-8", errors="ignore") if isinstance(v, (bytes, bytearray)) else str(v)

    def _parse_ft_search(raw):
        if not raw:
            return 0, []
        total = int(_to_str(raw[0]))
        docs = []
        i = 1
        while i < len(raw):
            if i + 1 >= len(raw):
                break
            fields_list = raw[i + 1]
            if not isinstance(fields_list, (list, tuple)):
                i += 2
                continue
            d = {}
            for k, v in zip(fields_list[0::2], fields_list[1::2]):
                d[_to_str(k)] = _to_str(v)
            if "xp" in d:
                try:
                    d["xp"] = int(d["xp"])
                except Exception:
                    pass
            docs.append(d)
            i += 2
        return total, docs

    def _parse_ft_aggregate(raw):
        if not raw:
            return 0, []
        total = int(_to_str(raw[0]))
        docs = []
        for row in raw[1:]:
            if not isinstance(row, (list, tuple)):
                continue
            d = {}
            for i in range(0, len(row), 2):
                if i + 1 >= len(row):
                    break
                k = _to_str(row[i])
                v = row[i + 1]
                d[k] = _to_str(v)
            if "xp" in d:
                try:
                    d["xp"] = int(d["xp"])
                except Exception:
                    pass
            docs.append(d)
        return total, docs

    page = max(0, int(page))
    page_size = max(1, int(page_size))
    offset = page * page_size

    # construir filtros base (usando exactamente @xp:[(1 +inf] según tu snippet)
    if xp_max is None:
        xp_filter = "@xp:[(1 +inf]"
    else:
        xp_filter = f"@xp:[(1 {int(xp_max)}]"

    parts = [xp_filter, "-@faction_code:{mythos}"]

    if traits:
        if isinstance(traits, str):
            traits = [traits]
        traits_clean = [t.strip() for t in traits if t and t.strip()]
        if traits_clean:
            parts.append(f"@traits:{{{'|'.join(traits_clean)}}}")

    base_query = " ".join(filter(None, parts)) if parts else "*"

    preferred = preferred_faction.strip() if preferred_faction else None
    if preferred and preferred.lower() == "mythos":
        preferred = None
    
    # sin preferida -> FT.SEARCH normal
    if not preferred:
        raw = r.execute_command(
            "FT.SEARCH",
            index_name,
            base_query,
            "RETURN", "4", "code", "name", "faction_code", "xp",
            "SORTBY", "xp", "ASC",
            "LIMIT", str(offset), str(page_size),
        )
        total, docs = _parse_ft_search(raw)
        return {
            "faction": preferred_faction,
            "total_cards": total,
            "page": page,
            "page_size": page_size,
            "results": docs,
            "method": "single_query",
        }

    # con preferida -> usar FT.AGGREGATE siguiendo tu ejemplo
    pref_escaped = preferred.replace("\\", "\\\\").replace("'", "\\'")
    case_expr = f"case(@faction_code == '{pref_escaped}', 1, 0)"

    raw = r.execute_command(
        "FT.AGGREGATE",
        index_name,
        base_query,
        "LOAD", "4", "@code", "@name", "@faction_code", "@xp",
        "APPLY", case_expr, "AS", "has_pref",
        "APPLY", "1 - @has_pref", "AS", "priority",
        "SORTBY", "2", "@priority", "ASC",
        "LIMIT", str(offset), str(page_size),
    )

    total, docs = _parse_ft_aggregate(raw)

    # eliminar auxiliares si aparecen
    for d in docs:
        d.pop("has_pref", None)
        d.pop("priority", None)

    return {
        "faction": preferred_faction,
        "total_cards": total,
        "page": page,
        "page_size": page_size,
        "results": docs,
        "method": "aggregate_priority",
    }

In [190]:
def search_upgrades(
    r,
    traits=None,
    preferred_faction=None,
    xp_max=None,
    page=0,
    page_size=5,
    index_name="cards-idx",
):
    page = max(0, int(page))
    page_size = max(1, int(page_size))
    offset = page * page_size

    def _norm_tag(value: str) -> str:
        return value.strip().lower()

    # Filtro de XP (>0 y opcionalmente <= xp_max)
    if xp_max is None:
        xp_filter = "@xp:[(0 +inf]"
    else:
        xp_filter = f"@xp:[(0 {int(xp_max)}]"

    parts = [xp_filter, "-@faction_code:{mythos}"]

    # Filtro de traits (OR)
    if traits:
        if isinstance(traits, str):
            traits = [traits]
        traits_clean = [t.strip() for t in traits if t and t.strip()]
        if traits_clean:
            tags = "|".join(_norm_tag(t) for t in traits_clean)
            parts.append(f"@traits:{{{tags}}}")

    base_query = " ".join(parts) if parts else "*"

    preferred = preferred_faction.strip().lower() if preferred_faction else None
    if preferred == "mythos":
        preferred = None

    # Caso sin facción preferida: una sola query
    if not preferred:
        raw = r.execute_command(
            "FT.SEARCH",
            index_name,
            base_query,
            "RETURN", "4", "code", "name", "faction_code", "xp",
            "SORTBY", "xp", "ASC",
            "LIMIT", str(offset), str(page_size),
        )
        total, docs = _parse_ft_search(raw)
        return {
            "faction": preferred_faction,
            "total_cards": total,
            "page": page,
            "page_size": page_size,
            "results": docs,
            "method": "single_query",
        }

    # 1) Conteo y página de facción preferida
    preferred_query = f"{base_query} @faction_code:{{{preferred}}}"

    raw_pref_count = r.execute_command(
        "FT.SEARCH",
        index_name,
        preferred_query,
        "LIMIT", "0", "0",
    )
    total_pref, _ = _parse_ft_search(raw_pref_count)

    raw_pref_page = r.execute_command(
        "FT.SEARCH",
        index_name,
        preferred_query,
        "RETURN", "4", "code", "name", "faction_code", "xp",
        "SORTBY", "xp", "ASC",
        "LIMIT", str(offset), str(page_size),
    )
    _, docs_pref = _parse_ft_search(raw_pref_page)

    # 2) Si faltan cartas para completar página, completar con otras facciones
    remaining = page_size - len(docs_pref)
    docs_other = []
    total_other = 0

    if remaining > 0:
        other_query = f"{base_query} -@faction_code:{{{preferred}}}"

        raw_other_count = r.execute_command(
            "FT.SEARCH",
            index_name,
            other_query,
            "LIMIT", "0", "0",
        )
        total_other, _ = _parse_ft_search(raw_other_count)

        # Offset en "otras facciones" según lo ya consumido por páginas previas
        if offset < total_pref:
            other_offset = 0
        else:
            other_offset = offset - total_pref

        raw_other_page = r.execute_command(
            "FT.SEARCH",
            index_name,
            other_query,
            "RETURN", "4", "code", "name", "faction_code", "xp",
            "SORTBY", "xp", "ASC",
            "LIMIT", str(other_offset), str(remaining),
        )
        _, docs_other = _parse_ft_search(raw_other_page)

    return {
        "faction": preferred_faction,
        "total_cards": total_pref + total_other,
        "page": page,
        "page_size": page_size,
        "results": docs_pref + docs_other,
        "method": "two_queries" if remaining > 0 else "preferred_only",
    }

In [191]:
search_upgrades(
    r,
    traits=["spell", "item"],
    preferred_faction="mystic",
    xp_max=5,
    page=0,
    page_size=5
)

{'faction': 'mystic',
 'total_cards': 667,
 'page': 0,
 'page_size': 5,
 'results': [{'_key': 'card:60420',
   'xp': '1',
   'name': 'Eldritch Inspiration',
   'faction_code': 'mystic',
   'code': '60420'},
  {'_key': 'card:60520',
   'xp': '1',
   'name': 'Cherished Keepsake',
   'faction_code': 'survivor',
   'code': '60520'},
  {'_key': 'card:60521',
   'xp': '1',
   'name': 'Leather Coat',
   'faction_code': 'survivor',
   'code': '60521'},
  {'_key': 'card:60120',
   'xp': '1',
   'name': 'Evidence!',
   'faction_code': 'guardian',
   'code': '60120'},
  {'_key': 'card:60121',
   'xp': '1',
   'name': 'Galvanize',
   'faction_code': 'guardian',
   'code': '60121'}],
 'method': 'preferred_only'}

### Tarea 3. ¿Cómo comprobarias que has implementado correctamente las búsquedas anteriores?

6) ¿Cómo comprobar que está bien implementado? (lo que te piden en el punto 3)
Te dejo un checklist “tipo entrega”:
Verificación A (facciones AND/OR)
Elige 2 facciones (ej.: ["mystic","guardian"]).
Ejecuta:
AND: todas las cartas devueltas deben tener ambas en faction_code
OR: todas deben tener al menos una
Puedes validarlo con un assert simple:

In [192]:
def check_A(results, factions, mode):
    factions = set(f.lower() for f in factions)
    for d in results["results"]:
        card_factions = set((d.get("faction_code") or "").split("|"))
        if mode == "AND":
            assert factions.issubset(card_factions)
        else:
            assert len(factions.intersection(card_factions)) > 0


Verificación B (traits por facción)
Coge una facción (ej. mystic).
Obtén top_traits_for_faction(...) para la primera página.
Valida la frecuencia comparando:
(i) tu resultado
(ii) un conteo manual en Python para esa facción (el propio fallback ya te hace esa validación indirecta).
Además, si usas FT.AGGREGATE, estás usando exactamente el mecanismo de “GROUP BY + COUNT” explicado en los PDFs.
Verificación C (xp>0, excluir mythos, xp_max y prioridad)
Para una consulta C concreta, verifica:
xp de cada resultado: int(xp) > 0 y <= xp_max si existe (rangos NUMERIC).
faction_code no contiene mythos (negación).
Si pones preferred_faction="seeker", los primeros resultados (antes de “rellenar”) pertenecen a esa facción (tu mezcla en 2 pasos lo garantiza).
Y un truco útil del temario: FT.SEARCH ... LIMIT 0 0 para comprobar el número total de aciertos sin traer documentos (te sirve para comparar “cuántas” devuelve tu filtro).

Ejemplo de uso

In [193]:
r = init_redis(host="localhost", port=6379)
ensure_cards_index(r)

# A) OR
res_or = search_by_factions(r, ["mystic", "guardian"], mode="OR", page=0)
# A) AND (multifacción)
res_and = search_by_factions(r, ["mystic", "guardian"], mode="AND", page=0)

# B) Top traits mystic (15)
top_traits = top_traits_for_faction(r, "mystic", page=0)

# C) Upgrades: xp>0, xp<=3, excluir mythos, preferir seeker, mezclar seeker+rogue
upg = search_upgrades(
    r,
    traits=["Weapon", "Spell"],       # ejemplo: OR
    preferred_faction="seeker",
    xp_max=3,
    page=0
)

# **Objetivo III**

El equipo del portal quiere implementar un recomendador automático de cartas. La idea es que, dada una carta, el sistema muestre otras 5 cartas parecidas a dicha carta que a los usuarios les pudieran interesar. Para ello, se ha consultado con el equipo de diseño de “Arkham Dread” y han dicho que lo mejor sería utilizar el nombre y el texto de la carta, si no fuera posible también indican que los traits son una alternativa, aunque son menos específicos generales. Al preguntarles por las ilustraciones, el equipo de diseño ha respondido que están más pensadas como decoración y no dan información sobre lo que hace la carta. Con esta información el equipo de desarrollo ha planteado 4 alternativas de más sencilla a más compleja:
- **A.** Usar solo metadatos de la carta y obviar el nombre y el texto: Proponen buscar cartas de
la misma facción y que tengan alguno de los traits de la carta sobre la que se realiza la
búsqueda.
- **B.** Capacidades full-text de redis: Aprovechar las capacidades de redis y hace una busqueda
full-text con el texto de la carta.
- **C.** Búsqueda semántica: Hacer embeddings con el nombre y el texto de la carta para luego
hacer búsqueda semántica.


### Tarea 1. Diseña un índice que permita realizar las alternativas anteriores.

In [194]:
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=False)

def crear_indice_objetivo3(index_name="idx_cards", dim=384):
    """
    Índice único para:
      A) TAG: faction_code, traits (multi-valor con '|')
      B) TEXT: name, text
      C) VECTOR: embedding (KNN)
    """
    try:
        r.execute_command(
            "FT.CREATE", index_name,
            "ON", "HASH",
            "PREFIX", "1", "card:",
            "SCHEMA",
            "faction_code", "TAG", "SEPARATOR", "|",
            "traits", "TAG", "SEPARATOR", "|",
            "name", "TEXT",
            "text", "TEXT",
            "embedding", "VECTOR", "HNSW", "6",
                "TYPE", "FLOAT32",
                "DIM", str(dim),
                "DISTANCE_METRIC", "COSINE"
        )
        print("Índice creado:", index_name)
    except Exception as e:
        print("No se pudo crear:", e)


In [195]:
crear_indice_objetivo3()

No se pudo crear: Index already exists


### Tarea 2. Haz una función python que implemente cada una de las alternativas propuestas. La función debe recibir el código de una carta e imprimir por pantalla 5 otras cartas parecidas.

In [196]:
import re
import numpy as np

def get_card_no_embedding(code: str) -> dict:
    """Lee card:<code> y devuelve dict (str->str), ignorando embedding (binario)."""
    d = r.hgetall(f"card:{code}")
    if not d:
        raise ValueError(f"No existe card:{code} en Redis")

    out = {}
    for k, v in d.items():
        ks = k.decode()

        # No intentamos decodificar embedding (binario), lo dejamos como bytes para no romperlo. 
        if ks == "embedding":
            continue

        out[ks] = v.decode(errors="ignore")  # por si hay algún carácter raro en text
    return out

def imprimir_docs(titulo, docs):
    print("\n" + "="*80)
    print(titulo)
    print("="*80)
    for i, doc in enumerate(docs, 1):
        extra = f" | score={doc['score']:.4f}" if "score" in doc else ""
        print(f"{i:02d}) [{doc.get('code')}] {doc.get('name')}{extra}")
        print(f"    Facción: {doc.get('faction_code')} | Traits: {doc.get('traits')} | Name: {doc.get('name')} | Text: {doc.get('text')}")
        display_card(doc.get("image_url", ""))

### A) Misma facción y comparte alguno de los traits

In [218]:
def recomendar_A(code: str, k: int = 5, index_name="idx_cards"):

    # Obtenermos los datos de la carta. 
    base = get_card_no_embedding(code)
    faction_raw = (base.get("faction_code") or "").strip()
    traits = [t for t in (base.get("traits") or "").split("|") if t]

    # En caso de que no tenga ni facción de traits, no podemos recomendar por metadatos. 
    if not faction_raw or not traits:
        print("No hay faction o traits en la carta base.")
        return

    # Si la carta base fuese multi-facción, buscaríamos cartas que compartan al menos una de esas facciones 
    # y al menos uno de esos traits.
    factions = [f for f in faction_raw.split("|") if f]
    faction_or = "|".join(factions)
    traits_query = " ".join([f"@traits:{{{t}}}" for t in traits])

    query = f"@faction_code:{{{faction_or}}} {traits_query} -@code:{{{code}}}"

    # Pedimos k
    res = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "5", "name", "faction_code", "traits", "type_code", "image_url",
        "LIMIT", "0", str(k)
    )

    # Parseamos resultado (total y lista de docs)
    total, docs = _parse_ft_search(res)

    print(f"Total candidatos: {total}")
    imprimir_docs("Recomendación por metadatos", docs)

### B) Full-text con el texto de la carta

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')

def recomendar_B(code: str, k: int = 5, index_name="idx_cards"):
    base = get_card_no_embedding(code)
    text = (base.get("text") or "")
    name = (base.get("name") or "")

    stop_words = set(stopwords.words('english'))
    
    s = (name + " " + text).lower()
    s = re.sub(r"[^a-z0-9áéíóúüñ\s]", " ", s)
    words = [w for w in s.split() if len(w) >= 4][:8]
    words = [w for w in words if w not in stop_words]

    if not words:
        print("B) No hay texto útil.")
        return

    or_part = "|".join(words)
    query = f"(@text:({or_part}) ) -@code:{{{code}}}" # Sale mejor si no incluimos el name en la búsqueda, porque suele ser muy específico y no aporta tanto a la similitud semántica.
    #     query = f"(@name:({or_part}) | @text:({or_part}) ) -@code:{{{code}}}"
    res = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "3", "name", "text", "image_url",
        "LIMIT", "0", str(k)
    )

    total, docs = _parse_ft_search(res)

    print(f"B) Total candidatos: {total}")
    imprimir_docs("B) Recomendación full-text", docs)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sheng\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### C) Búsqueda semántica (embeddings + KNN)

In [ ]:
from redis.commands.search.query import Query
import numpy as np

def recomendar_C(code: str, k: int = 5, index_name="idx_cards", pattern="card:*", batch=200):
    """
    C) Búsqueda semántica (embeddings + KNN)

    - Si faltan embeddings en Redis, los calcula y los guarda automáticamente (campo HASH 'embedding').
    - Luego ejecuta KNN sobre @embedding y devuelve k cartas similares.
    - Arregla:
        * UnicodeDecodeError (embedding es binario) -> get_card debe ignorar 'embedding'
        * [None] en code -> lo saca del doc.id (ej: "card:1029")
        * que se recomiende a sí misma -> filtra por code_found
    """

    # extraer code desde docid "card:XXXX"
    def _docid_to_code(docid: str) -> str:
        return docid.split(":", 1)[1] if isinstance(docid, str) and ":" in docid else docid

    # Asegurar embeddings en Redis (si faltan, los generamos)
    if not r.hexists(f"card:{code}", "embedding"):
        print("C) No hay embeddings en Redis todavía. Se procederá a generarlos.")

        pipe = r.pipeline()
        n = 0

        for key in r.scan_iter(pattern):
            if r.hexists(key, "embedding"):
                continue

            h = r.hgetall(key)
            name = h.get(b"name", b"").decode(errors="ignore")
            text = h.get(b"text", b"").decode(errors="ignore")
            s = (name + " " + text).strip()
            if not s:
                continue

            vec_bytes = create_embedding(s).tobytes()
            pipe.hset(key, mapping={"embedding": vec_bytes})
            n += 1

            if n % batch == 0:
                pipe.execute()

        pipe.execute()
        print(f"C) Embeddings guardados en {n} cartas (nuevos).")

    # 2) Construir vector de consulta desde la carta base
    base = get_card_no_embedding(code)
    text = (base.get("name", "") + " " + base.get("text", "")).strip()
    if not text:
        print("Devueltos: 0 (KNN)")
        print("La carta base no tiene texto/nombre suficiente.")
        imprimir_docs("Recomendación semántica (KNN)", [])
        return
    
    vec = create_embedding(text)

    # Pedimos k+1 por si aparece la propia carta (y la quitamos)
    q = (
        Query(f"*=>[KNN {k+1} @embedding $vec AS score]")
        .sort_by("score")  # COSINE: menor = más parecido
        .return_fields("code", "name", "score")
        .dialect(2)
    )

    res = r.ft(index_name).search(q, query_params={"vec": vec.tobytes()})

    docs = []
    for doc in res.docs:
        # doc.id suele ser "card:XXXX" (clave del HASH)
        docid = getattr(doc, "id", "")  # str
        code_found = getattr(doc, "code", None)
        url = getattr(doc, "image_url", None)

        # Si code no viene (None), lo sacamos del docid
        if code_found is None or str(code_found) == "None" or str(code_found).strip() == "":
            if docid:
                code_found = _docid_to_code(str(docid))

        # Excluir la carta base
        if str(code_found) == str(code):
            continue

        docs.append({
            "code": code_found,
            "name": getattr(doc, "name", None),
            "score": float(getattr(doc, "score", 0.0)),
        })

        if len(docs) >= k:
            break

    print(f"C) Devueltos: {len(docs)} (KNN)")
    imprimir_docs("C) Recomendación semántica (KNN)", docs)

### Tarea 3. Comprueba los resultados obtenidos ¿Qué diferencias hay entre las distintas alternativas? ¿Cuántas cartas es capaz de devolver cada aproximación? ¿Qué tal es el orden que devuelve? Da tu opinión justificada en los resultados obtenidos. Haz las pruebas usando la carta con código 1029 (shotgun).

In [254]:
code = "1029"

In [255]:
recomendar_A(code)

Total candidatos: 15

Recomendación por metadatos
01) [2226] Springfield M1903
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Springfield M1903 | Text: None


02) [2301] Lightning Gun
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Lightning Gun | Text: None


03) [03020] .32 Colt
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: .32 Colt | Text: None


04) [11032] Remington Model 1858
    Facción: guardian | Traits: Item|Weapon|Firearm | Name: Remington Model 1858 | Text: None


05) [5115] .45 Thompson
    Facción: guardian|rogue | Traits: Item|Weapon|Firearm|Illicit | Name: .45 Thompson | Text: None


In [256]:
recomendar_B(code) 

B) Total candidatos: 2304

B) Recomendación full-text
01) [08732] Cookie's Custom .32
    Facción: None | Traits: None | Name: Cookie's Custom .32 | Text: Fast. Uses (2 ammo). [action] Spend 1 ammo: Fight. Either fight with a base [combat] skill of 5, or get +2 [combat] for this attack. This attack deals +1 damage.


02) [7305] .25 Automatic
    Facción: None | Traits: None | Name: .25 Automatic | Text: Fast. Uses (4 ammo). [action] Spend 1 ammo: Fight. If the attacked enemy is exhausted, you get +2 [combat] and deal +1 damage for this attack. [reaction] After you evade an enemy at your location: Perform the above Fight ability without spending an action.


03) [08088] Old Shotgun
    Facción: None | Traits: None | Name: Old Shotgun | Text: Uses (0 ammo). While playing an event, treat Old Shotgun's uses value as 2. [action] Spend 1 ammo: Fight. You get +3 [combat] for this attack. Instead of its standard damage, this attack's damage is equal to the amount you succeed by, or fail by if you fail and would damage another investigator (to a minimum of 1, to a maximum of 3).


04) [08719] James "Cookie" Fredericks
    Facción: None | Traits: None | Name: James "Cookie" Fredericks | Text: Partner. Uses (3 ammo). [action] Spend 1 ammo and exhaust James "Cookie" Fredericks: Fight. Fight with a base [combat] skill of 5. If you succeed and the attacked enemy is non-[[Elite]], it cannot attack for the remainder of the round.


05) [53006] Colt Vest Pocket
    Facción: None | Traits: None | Name: Colt Vest Pocket | Text: Uses (5 ammo). [action] Spend 1 ammo: Fight. You get +2 [combat] for this attack. This attack deals +1 damage. Forced - At the end of the round, if you triggered Colt Vest Pocket's "[action]" ability: Discard it.


In [257]:
recomendar_C(code)

C) Devueltos: 5 (KNN)

C) Recomendación semántica (KNN)
01) [08088] Old Shotgun | score=0.1506
    Facción: None | Traits: None | Name: Old Shotgun | Text: None


02) [6327] Sawed-Off Shotgun | score=0.2253
    Facción: None | Traits: None | Name: Sawed-Off Shotgun | Text: None


03) [11032] Remington Model 1858 | score=0.3365
    Facción: None | Traits: None | Name: Remington Model 1858 | Text: None


04) [11022] Remington Model 1858 | score=0.3394
    Facción: None | Traits: None | Name: Remington Model 1858 | Text: None


05) [4273] Old Hunting Rifle | score=0.3866
    Facción: None | Traits: None | Name: Old Hunting Rifle | Text: None


### 1) ¿Qué diferencias hay entre las distintas alternativas?

* A (metadatos) recomienda usando facción + traits. Es básicamente un filtro por categorías del juego: si comparte esas etiquetas, entra. Es fácil de justificar, pero no “entiende” el contenido del texto.
* B (full-text) recomienda por palabras del nombre/texto. Funciona bien cuando hay coincidencias claras (por ejemplo “shotgun” o “fight”), pero si dos cartas son parecidas y no usan el mismo vocabulario, puede no detectarlo.
* C (semántica) recomienda por significado (embeddings + KNN). No necesita que coincidan palabras exactas: encuentra cartas parecidas por contexto/uso, y además aporta un score que permite ordenar mejor.

### 2) ¿Cuántas cartas es capaz de devolver cada aproximación?

Con la carta 1029 (Shotgun):
* A: encuentra 66 candidatos (es la que más devuelve).
* B: encuentra 15 candidatos (menos, porque depende del vocabulario).
* C: devuelve 5 porque le pedimos top-5 (KNN te puede devolver más si aumentas k; lo relevante es que genera un ranking por similitud continua).

### 3) ¿Qué tal es el orden que devuelve?

* A: el orden no es muy informativo. Al ser un filtro por etiquetas, el ranking no garantiza “más parecido” arriba y “menos parecido” abajo.
* B: el orden suele ser razonable, pero está guiado por relevancia textual: prioriza coincidencias de palabras (por eso salen varios “fighting/fight”).
* C: es el mejor ordenado, porque el ranking viene dado por el score (distancia coseno): cuanto menor, más similar. En tu salida, primero salen “Old Shotgun” y “Sawed-Off Shotgun”, que son claramente lo más cercano a “Shotgun”.

### Nuestra opinión justificada sobre los resultados obtenidos:

Con los resultados que hemos obtenido, se ve bastante claro qué aporta cada enfoque y dónde se queda corto.

* En **A (metadatos)**, el recomendador se apoya en información estructurada (facción y traits). Eso hace que sea una aproximación **muy estable y fácil de defender**, porque cada recomendación tiene una justificación directa: comparten etiquetas relevantes con la carta base. Además, en nuestro caso genera un conjunto amplio de candidatos (66), lo cual es útil si lo que queremos es  disponer d euna gama más amplia dentro de la misma familia de cartas. El punto débil es que, aunque el filtrado tiene sentido, **el orden de los resultados no refleja necesariamente el grado real de similitud**, ya que no estamos midiendo “cuánto” se parecen, sino solo si cumplen o no las condiciones del filtro.

* En **B (full-text)** el criterio cambia: aquí la similitud viene determinada por el **vocabulario** del nombre y del texto. En la práctica funciona bien cuando las cartas comparten términos muy característicos, y por eso aparecen coincidencias claras como *Old Shotgun*. Sin embargo, también se aprecia que el método tiende a priorizar cartas que repiten palabras frecuentes o destacadas del texto (por ejemplo, todo lo relacionado con *fight/fighting*). Esto no es incorrecto, pero sí implica que el recomendador está midiendo sobre todo **parecido léxico**, no necesariamente parecido funcional o temático. Por tanto, puede devolver cartas razonables, pero también puede dejar fuera opciones semánticamente cercanas si se describen con otra terminología.

* Por último, **C (búsqueda semántica)** es el enfoque que mejor captura la idea de “cartas parecidas” tal y como la entendería un usuario. Al trabajar con embeddings, la comparación se hace a nivel de significado y contexto, no de etiquetas ni de coincidencias exactas de palabras. Eso se nota en el top-5: aparecen principalmente armas muy próximas a una escopeta (otras escopetas y rifles), y además el **orden es coherente**, porque el score actúa como una medida continua de similitud (menor distancia → mayor parecido). En conjunto, es la alternativa que ofrece recomendaciones más parecidas con la carta base y con un ranking más convincente.

* En resumen: si buscamos una solución sencilla, interpretable y rápida de implementar, **A** cumple bien, aunque no priorice finamente los resultados. Si queremos aprovechar texto sin añadir modelos de machine learning, **B** es una opción razonable, pero su noción de parecido está condicionada por el vocabulario. Y si el objetivo es maximizar la calidad de las recomendaciones y el orden de salida, **C** es claramente la opción más sólida.
